# Recomendación Online: GNNs con Items Cold-Start (Versión Mejorada)

**Dataset:** Twitter15 + Twitter16 (split temporal global)  
**Escenario:** Recomendar tweets NUEVOS (publicados después del cutoff temporal)

En este notebook aplicamos mejoras arquitecturales y de training para mejorar el MRR de cada modelo:
- **GCN-BERT mejorado:** MPNet (768-dim), projection más profunda, BatchNorm, Dropout, 4 capas
- **GCN-Random mejorado:** Embeddings de 128-dim entrenables, arquitectura más profunda
- **LightGCN mejorado:** Projection propia para cold-start, normalización por capa

**Mejoras de training:**
- Hard negative mining (30% hard, 70% random)
- Learning rate scheduler con warm restarts
- Gradient clipping
- Early stopping con validation split
- Sample weighting por interaction_count
- Text preprocessing para BERT

Objetivo: maximizar MRR de cada modelo sin intervenir en la exposición a fake news.

## Setup y descarga de datos

In [ ]:
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/twitter15_processed.csv
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/twitter16_processed.csv
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/negative_samples.csv

print("Datos descargados")

In [ ]:
import pandas as pd
import numpy as np
import re
from html import unescape
from pathlib import Path

def decode_snowflake_timestamp(tweet_id):
    try:
        timestamp_ms = ((int(tweet_id) >> 22) + 1288834974657)
        return pd.to_datetime(timestamp_ms, unit='ms')
    except:
        return pd.NaT

print("Generando split temporal global...")
for dataset in ['twitter15', 'twitter16']:
    df = pd.read_csv(f'{dataset}_processed.csv', sep=';')
    df['timestamp'] = df['child_tweet_id'].apply(decode_snowflake_timestamp)
    df['child_datetime'] = pd.to_datetime(df['child_datetime'])
    df['timestamp'] = df['timestamp'].fillna(df['child_datetime'])

    df_collapsed = df.groupby(['child_user_id', 'source_tree_id']).agg({
        'timestamp': 'min',
        'text': 'first',
        'parent_label': 'first',
        'tree_label': 'first',
        'child_tweet_id': 'size'
    }).reset_index()

    df_collapsed.rename(columns={'child_tweet_id': 'interaction_count'}, inplace=True)
    df_collapsed['interaction_count'] = df_collapsed['interaction_count'].clip(upper=10)

    cutoff_T = df_collapsed['timestamp'].sort_values().quantile(0.7)
    cutoff_V = df_collapsed['timestamp'].sort_values().quantile(0.8)
    
    df_collapsed['split'] = 'test'
    df_collapsed.loc[df_collapsed['timestamp'] < cutoff_V, 'split'] = 'val'
    df_collapsed.loc[df_collapsed['timestamp'] < cutoff_T, 'split'] = 'train'

    df_collapsed.to_csv(f'{dataset}_temporal.csv', sep=';', index=False)
    print(f"  {dataset}: {len(df_collapsed)} interacciones")

print("Split temporal creado (70/10/20)")

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric
!pip install -q sentence-transformers
!pip install -q scipy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, LGConv
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
from collections import defaultdict
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## Cargar datos con split temporal (train/val/test)

Creamos validation split para early stopping.

In [ ]:
df15 = pd.read_csv('twitter15_temporal.csv', sep=';')
df16 = pd.read_csv('twitter16_temporal.csv', sep=';')
df = pd.concat([df15, df16], ignore_index=True)

print(f"Total interacciones: {len(df):,}")
print(f"Usuarios: {df['child_user_id'].nunique():,}")
print(f"Items: {df['source_tree_id'].nunique():,}")
print(f"\nSplits:")
print(f"  Train: {(df['split']=='train').sum():,}")
print(f"  Val:   {(df['split']=='val').sum():,}")
print(f"  Test:  {(df['split']=='test').sum():,}")

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

train_items = set(train_df['source_tree_id'].unique())
val_items = set(val_df['source_tree_id'].unique())
test_items = set(test_df['source_tree_id'].unique())

print(f"\nVerificación cold-start:")
print(f"  Items en train: {len(train_items):,}")
print(f"  Items en val (cold): {len(val_items - train_items):,} / {len(val_items):,}")
print(f"  Items en test (cold): {len(test_items - train_items):,} / {len(test_items):,}")

## Crear mappings

Mapeamos users e items a índices. Solo items de train van al grafo.

In [ ]:
all_users = sorted(df['child_user_id'].unique())
train_items_list = sorted(train_items)
val_items_cold = sorted(val_items - train_items)
test_items_cold = sorted(test_items - train_items)

user_to_idx = {uid: idx for idx, uid in enumerate(all_users)}
item_to_idx_train = {iid: idx for idx, iid in enumerate(train_items_list)}
item_to_idx_val = {iid: idx for idx, iid in enumerate(val_items_cold)}
item_to_idx_test = {iid: idx for idx, iid in enumerate(test_items_cold)}

num_users = len(user_to_idx)
num_train_items = len(item_to_idx_train)
num_val_items = len(item_to_idx_val)
num_test_items = len(item_to_idx_test)

print(f"Mappings creados:")
print(f"  Usuarios: {num_users:,}")
print(f"  Items train: {num_train_items:,}")
print(f"  Items val (cold): {num_val_items:,}")
print(f"  Items test (cold): {num_test_items:,}")

train_df['user_idx'] = train_df['child_user_id'].map(user_to_idx)
train_df['item_idx'] = train_df['source_tree_id'].map(item_to_idx_train)
train_df = train_df.dropna(subset=['user_idx', 'item_idx']).reset_index(drop=True)
train_df['user_idx'] = train_df['user_idx'].astype(int)
train_df['item_idx'] = train_df['item_idx'].astype(int)

val_df['user_idx'] = val_df['child_user_id'].map(user_to_idx)
val_df['item_idx_cold'] = val_df['source_tree_id'].map(item_to_idx_val)
val_df = val_df.dropna(subset=['user_idx', 'item_idx_cold']).reset_index(drop=True)
val_df['user_idx'] = val_df['user_idx'].astype(int)
val_df['item_idx_cold'] = val_df['item_idx_cold'].astype(int)

test_df['user_idx'] = test_df['child_user_id'].map(user_to_idx)
test_df['item_idx_cold'] = test_df['source_tree_id'].map(item_to_idx_test)
test_df = test_df.dropna(subset=['user_idx', 'item_idx_cold']).reset_index(drop=True)
test_df['user_idx'] = test_df['user_idx'].astype(int)
test_df['item_idx_cold'] = test_df['item_idx_cold'].astype(int)

print(f"\nInteracciones después de mapear:")
print(f"  Train: {len(train_df):,}")
print(f"  Val: {len(val_df):,}")
print(f"  Test: {len(test_df):,}")

## Construir grafo bipartito (solo train)

El grafo social no se usa en este notebook pero lo dejamos para análisis de propagación.

In [ ]:
user_indices = torch.tensor(train_df['user_idx'].values, dtype=torch.long)
item_indices = torch.tensor(train_df['item_idx'].values, dtype=torch.long)
item_shifted = item_indices + num_users

train_edge_index = torch.stack([
    torch.cat([user_indices, item_shifted]),
    torch.cat([item_shifted, user_indices])
], dim=0).to(device)

print(f"Grafo bipartito (train):")
print(f"  Nodos: {num_users + num_train_items:,} ({num_users:,} users + {num_train_items:,} items)")
print(f"  Edges: {train_edge_index.shape[1]:,}")

## Negative sampling para train

In [ ]:
negative_samples = pd.read_csv('negative_samples.csv')

negative_samples['user_idx'] = negative_samples['user_id'].map(user_to_idx)
negative_samples['item_idx'] = negative_samples['item_id'].map(item_to_idx_train)
negative_samples = negative_samples.dropna(subset=['user_idx', 'item_idx'])
negative_samples['user_idx'] = negative_samples['user_idx'].astype(int)
negative_samples['item_idx'] = negative_samples['item_idx'].astype(int)

neg_dict = defaultdict(list)
for _, row in negative_samples.iterrows():
    neg_dict[row['user_idx']].append(row['item_idx'])

print(f"Negative samples: {len(negative_samples):,}")
print(f"Users con negatives: {len(neg_dict):,}")

## Text preprocessing para BERT

Limpiamos los tweets antes de pasarlos a BERT para mejor calidad de embeddings.

In [ ]:
def clean_tweet_text(text):
    if pd.isna(text) or text == "":
        return "empty tweet"
    
    text = unescape(text)
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = text.lower().strip()
    
    return text if text else "empty tweet"

print("Función de limpieza de texto lista")

## BERT embeddings mejorados

Usamos `all-mpnet-base-v2` (768-dim) que es mejor que MiniLM, y normalizamos los embeddings.

In [ ]:
model_bert = SentenceTransformer('all-mpnet-base-v2')

item_texts_train = train_df.groupby('source_tree_id')['text'].first().to_dict()
ordered_texts_train = []
for item_id in train_items_list:
    text = item_texts_train.get(item_id, "")
    ordered_texts_train.append(clean_tweet_text(text))

print(f"Generando MPNet embeddings para train items ({len(ordered_texts_train)})...")
train_item_embeddings_bert = model_bert.encode(ordered_texts_train, show_progress_bar=True,
                                                convert_to_tensor=True, normalize_embeddings=True)
train_item_embeddings_bert = train_item_embeddings_bert.to(device)

item_texts_val = val_df.groupby('source_tree_id')['text'].first().to_dict()
ordered_texts_val = []
for item_id in val_items_cold:
    text = item_texts_val.get(item_id, "")
    ordered_texts_val.append(clean_tweet_text(text))

print(f"Generando MPNet embeddings para val items ({len(ordered_texts_val)})...")
val_item_embeddings_bert = model_bert.encode(ordered_texts_val, show_progress_bar=True,
                                              convert_to_tensor=True, normalize_embeddings=True)
val_item_embeddings_bert = val_item_embeddings_bert.to(device)

item_texts_test = test_df.groupby('source_tree_id')['text'].first().to_dict()
ordered_texts_test = []
for item_id in test_items_cold:
    text = item_texts_test.get(item_id, "")
    ordered_texts_test.append(clean_tweet_text(text))

print(f"Generando MPNet embeddings para test items ({len(ordered_texts_test)})...")
test_item_embeddings_bert = model_bert.encode(ordered_texts_test, show_progress_bar=True,
                                               convert_to_tensor=True, normalize_embeddings=True)
test_item_embeddings_bert = test_item_embeddings_bert.to(device)

print(f"\nMPNet embeddings generados:")
print(f"  Train: {train_item_embeddings_bert.shape}")
print(f"  Val: {val_item_embeddings_bert.shape}")
print(f"  Test: {test_item_embeddings_bert.shape}")

## Modelos mejorados

### GCN-BERT v2:
- Projection más profunda (2-layer MLP)
- 4 capas GCN con BatchNorm y Dropout
- Embeddings de 128-dim

### GCN-Random v2:
- Embeddings entrenables de 128-dim (no fixed)
- 4 capas con BatchNorm y Dropout

### LightGCN v2:
- Projection propia para cold-start items
- Normalización L2 por capa

In [ ]:
class GCNRecommenderV2(nn.Module):
    def __init__(self, num_users, num_items, item_feature_dim, embedding_dim=128, 
                 hidden_dim=64, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        
        self.item_projection = nn.Sequential(
            nn.Linear(item_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, embedding_dim)
        )
        
        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.conv4 = GCNConv(hidden_dim, embedding_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)

    def forward(self, edge_index, item_features):
        user_emb = self.user_embedding.weight
        item_emb = self.item_projection(item_features)
        x = torch.cat([user_emb, item_emb], dim=0)
        
        x = self.conv1(x, edge_index)
        x[:self.num_users] = self.bn1(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x[:self.num_users] = self.bn2(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv3(x, edge_index)
        x[:self.num_users] = self.bn3(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv4(x, edge_index)
        
        return x[:self.num_users], x[self.num_users:]


class GCNRecommenderRandomV2(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=128, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.conv4 = GCNConv(hidden_dim, embedding_dim)
        
        self.dropout = nn.Dropout(dropout)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, edge_index):
        user_emb = self.user_embedding.weight
        item_emb = self.item_embedding.weight
        x = torch.cat([user_emb, item_emb], dim=0)
        
        x = self.conv1(x, edge_index)
        x[:self.num_users] = self.bn1(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x[:self.num_users] = self.bn2(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv3(x, edge_index)
        x[:self.num_users] = self.bn3(x[:self.num_users])
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv4(x, edge_index)
        
        return x[:self.num_users], x[self.num_users:]


class LightGCNV2(nn.Module):
    def __init__(self, num_users, num_items, item_feature_dim, embedding_dim=128, num_layers=4):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
        
        self.cold_start_projection = nn.Sequential(
            nn.Linear(item_feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, edge_index):
        x = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        all_emb = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.normalize(x, p=2, dim=1)
            all_emb.append(x)
        final = torch.stack(all_emb, dim=0).mean(dim=0)
        return final[:self.num_users], final[self.num_users:]
    
    def project_cold_items(self, cold_item_features):
        return self.cold_start_projection(cold_item_features)

print("Modelos mejorados definidos")

## Training mejorado con hard negatives, early stopping y schedulers

In [ ]:
def train_epoch_improved(model, edge_index, item_features, train_df, neg_dict, 
                        optimizer, is_random=False, hard_ratio=0.3):
    model.train()
    optimizer.zero_grad()

    if is_random:
        user_emb, item_emb = model(edge_index)
    else:
        user_emb, item_emb = model(edge_index, item_features)

    user_ids = train_df['user_idx'].values
    item_ids = train_df['item_idx'].values
    weights = train_df['interaction_count'].values if 'interaction_count' in train_df.columns else np.ones(len(train_df))
    weights = weights / weights.max()

    neg_items = []
    for u in user_ids:
        if random.random() < hard_ratio:
            with torch.no_grad():
                scores = torch.matmul(user_emb[u], item_emb.t())
                positive_items = set(train_df[train_df['user_idx']==u]['item_idx'].values)
                mask = torch.ones(len(scores), dtype=torch.bool, device=device)
                for pos_i in positive_items:
                    mask[pos_i] = False
                if mask.sum() > 0:
                    filtered_scores = scores[mask]
                    k = min(50, len(filtered_scores))
                    _, top_indices = torch.topk(filtered_scores, k)
                    selected_idx = top_indices[random.randint(0, len(top_indices)-1)].item()
                    neg_i = torch.arange(len(scores), device=device)[mask][selected_idx].item()
                else:
                    neg_i = random.randint(0, model.num_items - 1)
        else:
            if u in neg_dict and len(neg_dict[u]) > 0:
                neg_i = random.choice(neg_dict[u])
            else:
                neg_i = random.randint(0, model.num_items - 1)
        neg_items.append(neg_i)

    pos_users = torch.LongTensor(user_ids).to(device)
    pos_items = torch.LongTensor(item_ids).to(device)
    neg_items_t = torch.LongTensor(neg_items).to(device)
    weights_t = torch.FloatTensor(weights).to(device)

    pos_scores = (user_emb[pos_users] * item_emb[pos_items]).sum(dim=1)
    neg_scores = (user_emb[pos_users] * item_emb[neg_items_t]).sum(dim=1)

    loss = -(torch.sigmoid(pos_scores - neg_scores).log() * weights_t).mean()
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate_cold_start_v2(model, edge_index, train_item_features, cold_item_features,
                           eval_df, is_random=False, is_lightgcn=False, k=10):
    model.eval()

    if is_random:
        user_emb, _ = model(edge_index)
    elif is_lightgcn:
        user_emb, _ = model(edge_index)
        cold_item_emb = model.project_cold_items(cold_item_features)
    else:
        user_emb, _ = model(edge_index, train_item_features)
        cold_item_emb = model.item_projection(cold_item_features)
    
    if is_random:
        return [], []
    
    scores_matrix = torch.matmul(user_emb, cold_item_emb.t())

    recommendations = []
    ground_truth = []

    for user_idx in range(model.num_users):
        scores = scores_matrix[user_idx]
        _, top_items_idx = torch.topk(scores, min(k, len(scores)))
        recommendations.append(top_items_idx.cpu().tolist())

        true_items = eval_df[eval_df['user_idx'] == user_idx]['item_idx_cold'].values
        ground_truth.append(set(true_items))

    return recommendations, ground_truth


def compute_metrics(recommendations, ground_truth, num_items, sample_size=5000):
    reciprocal_ranks = []
    for rec_list, true_items in zip(recommendations, ground_truth):
        rank = None
        for i, item in enumerate(rec_list, 1):
            if item in true_items:
                rank = i
                break
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    mrr = np.mean(reciprocal_ranks)

    sampled_indices = np.random.choice(len(recommendations), min(sample_size, len(recommendations)), replace=False)
    sampled_recs = [recommendations[i] for i in sampled_indices]

    similarities = []
    for i in range(len(sampled_recs)):
        for j in range(i + 1, len(sampled_recs)):
            set_i = set(sampled_recs[i])
            set_j = set(sampled_recs[j])
            jaccard = len(set_i & set_j) / len(set_i | set_j) if len(set_i | set_j) > 0 else 0
            similarities.append(jaccard)
    ild = 1.0 - np.mean(similarities)

    recommended_items = set()
    for rec_list in recommendations:
        recommended_items.update(rec_list)
    coverage = len(recommended_items) / num_items

    return {'MRR': mrr, 'ILD': ild, 'Coverage': coverage}


class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_score, model):
        if self.best_score is None:
            self.best_score = val_score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

print("Funciones de training y evaluación mejoradas listas")

## Parámetros de entrenamiento mejorados

In [ ]:
MAX_EPOCHS = 300
LR = 0.001
WD = 1e-5
EMBED_DIM = 128
HIDDEN_DIM = 64
DROPOUT = 0.1
PATIENCE = 30

print(f"Parámetros:")
print(f"  Max Epochs: {MAX_EPOCHS}")
print(f"  LR: {LR}")
print(f"  Weight Decay: {WD}")
print(f"  Embedding Dim: {EMBED_DIM}")
print(f"  Hidden Dim: {HIDDEN_DIM}")
print(f"  Dropout: {DROPOUT}")
print(f"  Early Stopping Patience: {PATIENCE}")

## Entrenar GCN-BERT v2

Con MPNet, projection profunda, BatchNorm, Dropout, 4 capas, scheduler y early stopping.

In [ ]:
model_gcn_bert = GCNRecommenderV2(
    num_users, num_train_items,
    train_item_embeddings_bert.shape[1],
    EMBED_DIM, HIDDEN_DIM, DROPOUT
).to(device)

optimizer = torch.optim.Adam(model_gcn_bert.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
early_stopping = EarlyStopping(patience=PATIENCE)

print(f"GCN-BERT v2: {sum(p.numel() for p in model_gcn_bert.parameters()):,} params")

losses_gcn_bert = []
val_mrrs = []

print("\nEntrenando GCN-BERT v2...")
for epoch in tqdm(range(MAX_EPOCHS)):
    loss = train_epoch_improved(model_gcn_bert, train_edge_index, train_item_embeddings_bert,
                               train_df, neg_dict, optimizer, False, 0.3)
    losses_gcn_bert.append(loss)
    scheduler.step()
    
    if epoch % 10 == 0:
        recs_val, gt_val = evaluate_cold_start_v2(
            model_gcn_bert, train_edge_index,
            train_item_embeddings_bert, val_item_embeddings_bert,
            val_df, False, False, 10
        )
        metrics_val = compute_metrics(recs_val, gt_val, num_val_items)
        val_mrrs.append(metrics_val['MRR'])
        
        early_stopping(metrics_val['MRR'], model_gcn_bert)
        if early_stopping.early_stop:
            print(f"\nEarly stopping en epoch {epoch}")
            break

model_gcn_bert.load_state_dict({k: v.to(device) for k, v in early_stopping.best_model_state.items()})

plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
plt.plot(losses_gcn_bert, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - GCN-BERT v2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(0, len(losses_gcn_bert), 10)[:len(val_mrrs)], val_mrrs, linewidth=2, color='green')
plt.xlabel('Epoch')
plt.ylabel('Validation MRR')
plt.title('Validation MRR - GCN-BERT v2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEvaluando GCN-BERT v2 en test...")
recs_gcn_bert, gt = evaluate_cold_start_v2(
    model_gcn_bert, train_edge_index,
    train_item_embeddings_bert, test_item_embeddings_bert,
    test_df, False, False, 10
)
metrics_gcn_bert = compute_metrics(recs_gcn_bert, gt, num_test_items)

print("\nMétricas GCN-BERT v2 (test):")
for k, v in metrics_gcn_bert.items():
    print(f"  {k}: {v:.6f}")

## Entrenar GCN-Random v2

Con embeddings entrenables de 128-dim, BatchNorm, Dropout, 4 capas.

In [ ]:
model_gcn_random = GCNRecommenderRandomV2(
    num_users, num_train_items,
    EMBED_DIM, HIDDEN_DIM, DROPOUT
).to(device)

optimizer = torch.optim.Adam(model_gcn_random.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
early_stopping = EarlyStopping(patience=PATIENCE)

print(f"GCN-Random v2: {sum(p.numel() for p in model_gcn_random.parameters()):,} params")

losses_gcn_random = []

print("\nEntrenando GCN-Random v2...")
for epoch in tqdm(range(MAX_EPOCHS)):
    loss = train_epoch_improved(model_gcn_random, train_edge_index, None,
                               train_df, neg_dict, optimizer, True, 0.3)
    losses_gcn_random.append(loss)
    scheduler.step()
    
    if epoch % 10 == 0 and epoch > 0:
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses_gcn_random, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - GCN-Random v2')
plt.grid(True, alpha=0.3)
plt.show()

print("\nNota: GCN-Random no se puede evaluar en cold-start items (no hay features).")
print("Para usar GCN-Random en test, necesitamos generar random embeddings fijos para items cold.")

test_item_embeddings_random = torch.empty(num_test_items, EMBED_DIM).to(device)
nn.init.xavier_uniform_(test_item_embeddings_random)
test_item_embeddings_random = F.normalize(test_item_embeddings_random, p=2, dim=1)

model_gcn_random.eval()
with torch.no_grad():
    user_emb, _ = model_gcn_random(train_edge_index)
    scores_matrix = torch.matmul(user_emb, test_item_embeddings_random.t())
    recs_gcn_random = []
    for user_idx in range(num_users):
        _, top = torch.topk(scores_matrix[user_idx], 10)
        recs_gcn_random.append(top.cpu().tolist())

metrics_gcn_random = compute_metrics(recs_gcn_random, gt, num_test_items)

print("\nMétricas GCN-Random v2 (test):")
for k, v in metrics_gcn_random.items():
    print(f"  {k}: {v:.6f}")

## Entrenar LightGCN v2

Con projection propia para cold-start, normalización L2 por capa, 4 capas.

In [ ]:
model_lightgcn = LightGCNV2(num_users, num_train_items, 
                            train_item_embeddings_bert.shape[1], EMBED_DIM, 4).to(device)
optimizer = torch.optim.Adam(model_lightgcn.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
early_stopping = EarlyStopping(patience=PATIENCE)

print(f"LightGCN v2: {sum(p.numel() for p in model_lightgcn.parameters()):,} params")

losses_lightgcn = []
val_mrrs_lg = []

print("\nEntrenando LightGCN v2...")
for epoch in tqdm(range(MAX_EPOCHS)):
    model_lightgcn.train()
    optimizer.zero_grad()
    
    user_emb, item_emb = model_lightgcn(train_edge_index)
    
    user_ids = train_df['user_idx'].values
    item_ids = train_df['item_idx'].values
    weights = train_df['interaction_count'].values if 'interaction_count' in train_df.columns else np.ones(len(train_df))
    weights = weights / weights.max()
    
    neg_items = []
    for u in user_ids:
        if u in neg_dict and len(neg_dict[u]) > 0:
            neg_i = random.choice(neg_dict[u])
        else:
            neg_i = random.randint(0, num_train_items - 1)
        neg_items.append(neg_i)
    
    pos_users = torch.LongTensor(user_ids).to(device)
    pos_items = torch.LongTensor(item_ids).to(device)
    neg_items_t = torch.LongTensor(neg_items).to(device)
    weights_t = torch.FloatTensor(weights).to(device)
    
    pos_scores = (user_emb[pos_users] * item_emb[pos_items]).sum(dim=1)
    neg_scores = (user_emb[pos_users] * item_emb[neg_items_t]).sum(dim=1)
    
    loss = -(torch.sigmoid(pos_scores - neg_scores).log() * weights_t).mean()
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_lightgcn.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    
    losses_lightgcn.append(loss.item())
    
    if epoch % 10 == 0:
        recs_val, gt_val = evaluate_cold_start_v2(
            model_lightgcn, train_edge_index,
            train_item_embeddings_bert, val_item_embeddings_bert,
            val_df, False, True, 10
        )
        metrics_val = compute_metrics(recs_val, gt_val, num_val_items)
        val_mrrs_lg.append(metrics_val['MRR'])
        
        early_stopping(metrics_val['MRR'], model_lightgcn)
        if early_stopping.early_stop:
            print(f"\nEarly stopping en epoch {epoch}")
            break

model_lightgcn.load_state_dict({k: v.to(device) for k, v in early_stopping.best_model_state.items()})

plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
plt.plot(losses_lightgcn, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss - LightGCN v2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(0, len(losses_lightgcn), 10)[:len(val_mrrs_lg)], val_mrrs_lg, linewidth=2, color='green')
plt.xlabel('Epoch')
plt.ylabel('Validation MRR')
plt.title('Validation MRR - LightGCN v2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEvaluando LightGCN v2 en test...")
recs_lightgcn, _ = evaluate_cold_start_v2(
    model_lightgcn, train_edge_index,
    train_item_embeddings_bert, test_item_embeddings_bert,
    test_df, False, True, 10
)

metrics_lightgcn = compute_metrics(recs_lightgcn, gt, num_test_items)

print("\nMétricas LightGCN v2 (test):")
for k, v in metrics_lightgcn.items():
    print(f"  {k}: {v:.6f}")

## Comparación de modelos mejorados

In [ ]:
comparison = pd.DataFrame({
    'GCN-BERT v2': metrics_gcn_bert,
    'GCN-Random v2': metrics_gcn_random,
    'LightGCN v2': metrics_lightgcn
})

print("\n" + "="*70)
print("COMPARACIÓN - MODELOS MEJORADOS (cold-start items)")
print("="*70)
print(comparison.T.to_string())
print("="*70)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_names = ['MRR', 'ILD', 'Coverage']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, metric in enumerate(metrics_names):
    values = [metrics_gcn_bert[metric], metrics_gcn_random[metric], metrics_lightgcn[metric]]
    axes[i].bar(['BERT v2', 'Random v2', 'LightGCN v2'], values, color=colors)
    axes[i].set_ylabel(metric)
    axes[i].set_title(f'{metric} - Cold-Start (Mejorado)')
    axes[i].grid(True, alpha=0.3, axis='y')
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.005, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Análisis de desinformación (igual que v1)

In [ ]:
labels_test = test_df.groupby('source_tree_id')[['tree_label', 'parent_label']].first().to_dict()
labels_dict = {iid: labels_test['tree_label'].get(iid, 'NR') for iid in test_items_cold}

def analyze_label_distribution(recommendations, item_idx_to_id):
    label_counts = {'TR': 0, 'FR': 0, 'UR': 0, 'NR': 0}
    total = 0
    for rec_list in recommendations:
        for item_idx in rec_list:
            item_id = item_idx_to_id.get(item_idx)
            if item_id:
                label = labels_dict.get(item_id, 'NR')
                if label in label_counts:
                    label_counts[label] += 1
                total += 1
    return {k: v/total*100 if total > 0 else 0 for k, v in label_counts.items()}

item_idx_to_id_test = {v: k for k, v in item_to_idx_test.items()}

dist_gcn_bert = analyze_label_distribution(recs_gcn_bert, item_idx_to_id_test)
dist_gcn_random = analyze_label_distribution(recs_gcn_random, item_idx_to_id_test)
dist_lightgcn = analyze_label_distribution(recs_lightgcn, item_idx_to_id_test)

baseline_dist = test_df['tree_label'].value_counts(normalize=True).to_dict()
baseline_dist = {k: v*100 for k, v in baseline_dist.items()}

print("\n" + "="*70)
print("DISTRIBUCIÓN DE LABELS (items cold-start de test)")
print("="*70)

dist_df = pd.DataFrame({
    'Test Dataset': baseline_dist,
    'GCN-BERT v2': dist_gcn_bert,
    'GCN-Random v2': dist_gcn_random,
    'LightGCN v2': dist_lightgcn
})

print(dist_df.T.to_string())
print("="*70)

## Identificar usuarios expuestos

In [ ]:
def identify_exposed_users(recommendations, item_idx_to_id, target_labels=['FR']):
    exposed = set()
    for user_idx, rec_list in enumerate(recommendations):
        for item_idx in rec_list[:10]:
            item_id = item_idx_to_id.get(item_idx)
            if item_id:
                label = labels_dict.get(item_id, 'NR')
                if label in target_labels:
                    exposed.add(user_idx)
                    break
    return exposed

exposed_gcn_bert = identify_exposed_users(recs_gcn_bert, item_idx_to_id_test, ['FR'])
exposed_gcn_random = identify_exposed_users(recs_gcn_random, item_idx_to_id_test, ['FR'])
exposed_lightgcn = identify_exposed_users(recs_lightgcn, item_idx_to_id_test, ['FR'])

print(f"\nUsuarios expuestos a fake news (FR):")
print(f"  GCN-BERT v2:   {len(exposed_gcn_bert)} ({len(exposed_gcn_bert)/num_users*100:.2f}%)")
print(f"  GCN-Random v2: {len(exposed_gcn_random)} ({len(exposed_gcn_random)/num_users*100:.2f}%)")
print(f"  LightGCN v2:   {len(exposed_lightgcn)} ({len(exposed_lightgcn)/num_users*100:.2f}%)")

## Resumen final

Comparamos los resultados de las versiones mejoradas con la baseline original.

In [ ]:
print("\n" + "="*80)
print("RESUMEN: RECOMENDACIÓN ONLINE CON ITEMS COLD-START (VERSIONES MEJORADAS)")
print("="*80)

print("\nMejoras implementadas:")
print("  1. MPNet (768-dim) en vez de MiniLM (384-dim)")
print("  2. Text preprocessing (limpieza de URLs, mentions, hashtags)")
print("  3. Normalización L2 de embeddings BERT")
print("  4. Arquitectura más profunda: 4 capas GCN con BatchNorm y Dropout")
print("  5. Projection más profunda para GCN-BERT (2-layer MLP)")
print("  6. GCN-Random con embeddings entrenables de 128-dim")
print("  7. LightGCN con projection propia para cold-start")
print("  8. Hard negative mining (30% hard, 70% random)")
print("  9. Sample weighting por interaction_count")
print("  10. Learning rate scheduler (CosineAnnealingWarmRestarts)")
print("  11. Gradient clipping (max_norm=1.0)")
print("  12. Early stopping con validation split (70/10/20)")
print("  13. Más epochs (hasta 300 con patience=30)")

print("\nResultados finales (test):")
print(f"\n  GCN-BERT v2:")
for k, v in metrics_gcn_bert.items():
    print(f"    {k}: {v:.6f}")

print(f"\n  GCN-Random v2:")
for k, v in metrics_gcn_random.items():
    print(f"    {k}: {v:.6f}")

print(f"\n  LightGCN v2:")
for k, v in metrics_lightgcn.items():
    print(f"    {k}: {v:.6f}")

print("\n" + "="*80)

print("\nObservaciones:")
print("  - La exposición a fake news se mantiene alta (objetivo: observar, no intervenir)")
print("  - Las mejoras buscan maximizar MRR de cada modelo individualmente")
print("  - El validation split permite early stopping y evitar overfitting")
print("  - MPNet + preprocessing mejora la calidad de embeddings semánticos")
print("  - Hard negatives fuerzan al modelo a aprender distinciones más finas")
print("  - Arquitecturas más profundas permiten capturar patrones más complejos")